# Signal Hunters - Team Cyclops: final stack

CPU only, no internet. Inputs: the Round 2 data, the output of the main notebook (features,
LightGBM, CNNs, EfficientNets, repo code) and the output of the extra-models notebook. All base
models' out-of-fold and test probabilities are stacked by the same second-level LightGBM, then the
per-class bias is applied. Final file: `submissions/22_stackall_bias.csv`.

In [ ]:
import glob, os, shutil, subprocess
hits = glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)
os.environ['SIG_DATA'] = os.path.dirname(hits[0])
roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/oof/lgbm4_oof.csv', recursive=True)]
assert roots, 'Add the main notebook output as an input'
main = roots[0]
shutil.copytree(main, '/kaggle/working/signal-hunters', dirs_exist_ok=True)
os.environ['SIG_FEATURES'] = glob.glob('/kaggle/input/**/features/train_deg.parquet', recursive=True)[0].rsplit('/', 1)[0]
for p in glob.glob('/kaggle/input/**/oof/*.csv', recursive=True):
    dst = '/kaggle/working/signal-hunters/oof/' + os.path.basename(p)
    if not os.path.exists(dst):
        shutil.copy(p, dst)
        print('added', os.path.basename(p))
os.chdir('/kaggle/working/signal-hunters')
print(main, os.environ['SIG_FEATURES'])

## Stack every base model, then the per-class bias

In [ ]:
models = [m for m in ['lgbm4', 'cnn', 'cnn_s7', 'cnn_lvl', 'eff_lvl', 'effstk_lvl', 'effb2_lvl', 'eff_lvl_e20',
                     'eff_lvl_s11', 'effstk_lvl_s7', 'res34_lvl']
          if os.path.exists(f'oof/{m}_oof.csv') and os.path.exists(f'oof/{m}_test.csv')]
print('stacking', models)
subprocess.run(['python', 'scripts/stack.py', '--models', *models, '--features', 'fit_', 'trk_', 'lvl_',
                '--name', 'stackall'], check=True)
!python scripts/make_submission.py --models stackall --bias --out 22_stackall_bias
!ls -la submissions

## Blends of the stack with the strongest single models (log-probability weights, then bias)

In [ ]:
!python scripts/make_submission.py --models stackall eff_lvl_e20 res34_lvl effb2_lvl --weights 0.55 0.15 0.15 0.15 --bias --out 23_blend_top3
!python scripts/make_submission.py --models stackall stack7 --weights 0.5 0.5 --bias --out 24_blend_stack7
!python scripts/make_submission.py --models stackall eff_lvl_e20 res34_lvl --weights 0.7 0.15 0.15 --bias --out 25_blend_top2

## Pseudo-labelling: LightGBM and the stacker also train on test cadences the stack is at least 90% sure of

The input notebooks predate the `--pseudo` option, so the two scripts are rewritten from the repo first.

In [ ]:
%%writefile scripts/train_lgbm.py
"""LightGBM on the hand-built features, 5 folds from folds.csv.

Trains on original + PKS-degraded train rows, early-stops on the degraded validation fold (the
closest thing to test we have), and writes:
    oof/<name>_oof.csv       degraded-train OOF probabilities (used for blending and tuning)
    oof/<name>_oof_orig.csv  original-train OOF probabilities
    oof/<name>_test.csv      test probabilities, mean over folds
    python scripts/train_lgbm.py --name lgbm

--pseudo MODEL adds the unlabelled test cadences whose oof/MODEL_test.csv top probability is at
least --pseudo-thr to every fold's training set, labelled with that top class (pseudo-labelling:
the trees see real PKS feature distributions). Validation stays on labelled train rows only.
"""

import argparse
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, FEATURES, LABELS, ROOT, SEED, load_folds, load_split  # noqa: E402

PARAMS = dict(
    objective="multiclass", num_class=len(LABELS), learning_rate=0.05, num_leaves=31,
    min_data_in_leaf=40, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=1.0, n_jobs=CPUS, verbose=-1, seed=SEED,
)


def save(path: Path, ids: pd.Series, p: np.ndarray) -> None:
    pd.concat([ids.reset_index(drop=True), pd.DataFrame(p, columns=LABELS)], axis=1).to_csv(path, index=False)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="lgbm")
    ap.add_argument("--no-orig", action="store_true", help="train on degraded rows only")
    ap.add_argument("--pseudo", default=None, help="model whose test probabilities give pseudo-labels")
    ap.add_argument("--pseudo-thr", type=float, default=0.9)
    args = ap.parse_args()

    tr, te = load_split("train"), load_split("test")
    f_tr = pd.read_parquet(FEATURES / "train.parquet")
    f_dg = pd.read_parquet(FEATURES / "train_deg.parquet")
    f_te = pd.read_parquet(FEATURES / "test.parquet")
    y, folds = tr.y, load_folds()
    n = len(y)
    x_ps, y_ps = f_te.iloc[:0], np.zeros(0, dtype=int)
    if args.pseudo:
        p = pd.read_csv(ROOT / f"oof/{args.pseudo}_test.csv")[LABELS].to_numpy()
        keep = p.max(1) >= args.pseudo_thr
        x_ps, y_ps = f_te[keep].reset_index(drop=True), p[keep].argmax(1)
        print(f"pseudo-labels: {keep.sum()} test rows at >= {args.pseudo_thr}", flush=True)

    oof_dg = np.zeros((n, len(LABELS)))
    oof_tr = np.zeros((n, len(LABELS)))
    test = np.zeros((len(f_te), len(LABELS)))
    for k in range(5):
        fit = folds != k
        xs = [f_dg[fit]] if args.no_orig else [f_tr[fit], f_dg[fit]]
        x_fit = pd.concat(xs + [x_ps], ignore_index=True)
        y_fit = np.concatenate([y[fit]] * len(xs) + [y_ps])
        d_fit = lgb.Dataset(x_fit, y_fit, weight=compute_sample_weight("balanced", y_fit))
        d_val = lgb.Dataset(f_dg[~fit], y[~fit], weight=compute_sample_weight("balanced", y[~fit]))
        m = lgb.train(PARAMS, d_fit, 3000, valid_sets=[d_val],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_dg[~fit] = m.predict(f_dg[~fit], num_iteration=m.best_iteration)
        oof_tr[~fit] = m.predict(f_tr[~fit], num_iteration=m.best_iteration)
        test += m.predict(f_te, num_iteration=m.best_iteration) / 5
        print(f"fold {k}: iters {m.best_iteration}  degraded F1 "
              f"{f1_score(y[~fit], oof_dg[~fit].argmax(1), average='macro'):.4f}", flush=True)

    print(f"OOF macro-F1  original {f1_score(y, oof_tr.argmax(1), average='macro'):.4f}  "
          f"degraded {f1_score(y, oof_dg.argmax(1), average='macro'):.4f}")
    print(classification_report(y, oof_dg.argmax(1), target_names=LABELS, digits=3))
    cm = pd.DataFrame(confusion_matrix(y, oof_dg.argmax(1)), index=LABELS, columns=[c[:6] for c in LABELS])
    print("confusion (rows true, cols predicted):\n" + cm.to_string())
    print("test predicted mix:", pd.Series(np.array(LABELS)[test.argmax(1)]).value_counts().to_dict())

    imp = pd.Series(m.feature_importance("gain"), index=f_tr.columns).sort_values(ascending=False)
    print("top features:", imp.head(25).round(0).to_dict())
    (ROOT / "oof").mkdir(exist_ok=True)
    save(ROOT / f"oof/{args.name}_oof.csv", tr.meta["id"], oof_dg)
    save(ROOT / f"oof/{args.name}_oof_orig.csv", tr.meta["id"], oof_tr)
    save(ROOT / f"oof/{args.name}_test.csv", te.meta["id"], test)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/stack.py
"""Second-level LightGBM on the base models' OOF probabilities (plus chosen raw features).

Same folds as the base models: the stacker for fold k trains on the other folds' OOF rows and
predicts fold k, so its own OOF score is honest. Test predictions are the mean over the 5 fold
stackers applied to the base models' test probabilities.
    python scripts/stack.py --models lgbm3 cnn cnn_s7 --features fit_ trk_ --name stack
"""

import argparse
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, FEATURES, LABELS, ROOT, SEED, load_folds, load_split  # noqa: E402

PARAMS = dict(
    objective="multiclass", num_class=len(LABELS), learning_rate=0.03, num_leaves=15,
    min_data_in_leaf=60, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=5.0, n_jobs=CPUS, verbose=-1, seed=SEED,
)


def inputs(models: list[str], prefixes: list[str], part: str, feat_file: str) -> pd.DataFrame:
    cols = []
    for m in models:
        p = pd.read_csv(ROOT / f"oof/{m}_{part}.csv")[LABELS]
        cols.append(np.log(p.clip(1e-6, 1)).add_prefix(f"{m}__"))
    if prefixes:
        f = pd.read_parquet(FEATURES / feat_file)
        cols.append(f[[c for c in f.columns if c.startswith(tuple(prefixes))]].reset_index(drop=True))
    return pd.concat(cols, axis=1)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--features", nargs="*", default=[], help="raw feature name prefixes to add")
    ap.add_argument("--name", default="stack")
    ap.add_argument("--pseudo", default=None, help="model whose test probabilities give pseudo-labels")
    ap.add_argument("--pseudo-thr", type=float, default=0.9)
    args = ap.parse_args()

    tr, te = load_split("train"), load_split("test")
    y, folds = tr.y, load_folds()
    x = inputs(args.models, args.features, "oof", "train_deg.parquet")
    x_te = inputs(args.models, args.features, "test", "test.parquet")
    x_ps, y_ps = x_te.iloc[:0], np.zeros(0, dtype=int)
    if args.pseudo:
        p = pd.read_csv(ROOT / f"oof/{args.pseudo}_test.csv")[LABELS].to_numpy()
        keep = p.max(1) >= args.pseudo_thr
        x_ps, y_ps = x_te[keep], p[keep].argmax(1)
        print(f"pseudo-labels: {keep.sum()} test rows at >= {args.pseudo_thr}", flush=True)
    oof = np.zeros((len(y), len(LABELS)))
    test = np.zeros((len(x_te), len(LABELS)))
    for k in range(5):
        fit = folds != k
        x_fit, y_fit = pd.concat([x[fit], x_ps]), np.concatenate([y[fit], y_ps])
        d_fit = lgb.Dataset(x_fit, y_fit, weight=compute_sample_weight("balanced", y_fit))
        d_val = lgb.Dataset(x[~fit], y[~fit], weight=compute_sample_weight("balanced", y[~fit]))
        m = lgb.train(PARAMS, d_fit, 3000, valid_sets=[d_val], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[~fit] = m.predict(x[~fit], num_iteration=m.best_iteration)
        test += m.predict(x_te, num_iteration=m.best_iteration) / 5
        print(f"fold {k}: iters {m.best_iteration}  F1 {f1_score(y[~fit], oof[~fit].argmax(1), average='macro'):.4f}", flush=True)
    print(f"stack OOF macro-F1 (degraded): {f1_score(y, oof.argmax(1), average='macro'):.4f}  inputs {x.shape[1]}")
    print(classification_report(y, oof.argmax(1), target_names=LABELS, digits=3))
    pd.concat([tr.meta[["id"]], pd.DataFrame(oof, columns=LABELS)], axis=1).to_csv(ROOT / f"oof/{args.name}_oof.csv", index=False)
    pd.concat([te.meta[["id"]], pd.DataFrame(test, columns=LABELS)], axis=1).to_csv(ROOT / f"oof/{args.name}_test.csv", index=False)


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/train_lgbm.py --name lgbm_pl --pseudo stackall --pseudo-thr 0.9
!python scripts/stack.py --models lgbm_pl cnn cnn_s7 cnn_lvl eff_lvl effstk_lvl effb2_lvl eff_lvl_e20 eff_lvl_s11 effstk_lvl_s7 res34_lvl --features fit_ trk_ lvl_ --name stackpl --pseudo stackall --pseudo-thr 0.9
!python scripts/make_submission.py --models stackpl --bias --out 26_stackpl_bias